# 09 - Automatische FESR-Pipeline: Aceton [ppb]

Die Anzahl und Grenzen der Signalabschnitte werden jetzt **gelernt**, nicht vorgegeben.
`AutomaticSegments.fit` berechnet den Median der Trainingszyklen und unterteilt ihn so lange,
wie der Gewinn an linearer Rekonstruktion den BIC-artigen Komplexitaetspreis uebersteigt.
Die Mindestlaenge von acht Samples ist eine Aufloesungsgrenze, keine Segmentzahl.

Das ist ein greedy Verfahren mit einem automatischen Abbruchkriterium, keine Garantie einer global
optimalen Segmentierung. Mittelwert und Steigung jedes gelernten Abschnitts bilden die Merkmale.
Die Grenzen bleiben bei `transform` fest: neue Messungen werden nicht separat neu segmentiert.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p/'Networks'/'TCOCNNv3.py').exists())
for folder in ['Networks','Evaluation Seminar/Day_02','Evaluation Seminar/Day_03','Evaluation Seminar/Day_04']:
    sys.path.insert(0,str(ROOT/folder))
import numpy as np
import matplotlib.pyplot as plt
from day3_utils import plot_comparison, regression_metrics, show_results

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import GridSearchCV, GroupKFold
from dataset_pipeline import prepare_splits
from automatic_fesr import AutomaticSegments, FractionSelector
splits=prepare_splits('stored'); GAS='acetone'
X=splits['train']['X']; y=splits['train']['targets'][GAS]; groups=splits['train']['targets']['range']
extractor=AutomaticSegments().fit(X)
features=extractor.transform(X)
print('Automatisch gelernte Segmente:',extractor.n_segments_,'; Merkmale:',features.shape[1])
print('Train:',len(np.unique(groups)),'UGMs /',len(y),'Zyklen')

## Automatische Aufteilung und Abbruch

Orange zeigt die lineare Rekonstruktion in den gelernten Abschnitten. Die rechte Grafik zeigt den
Komplexitaetswert auf dem akzeptierten Suchpfad. Die Segmentzahl wird in **jedem CV-Trainingsfold neu
gelernt**. Weder Validierungs- noch Testzyklen bestimmen den Median oder die Abschnittsgrenzen.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(15,5)); time=np.arange(X.shape[-1])/10
axes[0].plot(time,extractor.reference_,label='Trainingsmedian')
for i,(a,b) in enumerate(zip(extractor.boundaries_[:-1],extractor.boundaries_[1:])):
    t=time[a:b]; centered=t-t.mean(); values=extractor.reference_[a:b]
    fitted=values.mean()+centered*(values@centered)/(centered@centered)
    axes[0].plot(t,fitted,color='tab:orange')
axes[0].set(title=f'{extractor.n_segments_} automatisch gelernte Abschnitte',xlabel='Zeit [s]',ylabel='Gespeicherter Signalwert')
path=np.asarray(extractor.path_); axes[1].plot(path[:,0],path[:,1],marker='.')
axes[1].set(xlabel='Akzeptierte Segmentzahl',ylabel='BIC-artiger Komplexitaetswert',title='Stopp ohne vorgegebenes n_seg')
plt.tight_layout(); plt.show()

## sklearn-Pipeline: alles innerhalb des Trainingsfolds lernen

`AutomaticSegments -> StandardScaler -> FractionSelector -> PLSRegression`.
Da die Zahl der Merkmale vom Fold abhaengt, waehlt die Feature Selection einen Anteil statt einer
festen Anzahl. Mindestens zwei Merkmale bleiben fuer die getesteten ein oder zwei PLS-Komponenten.
Pearson und RFE/Ridge werden ueber gruppierte Kreuzvalidierung auf Train verglichen; die separate
Validierung bestimmt die finale Variante. Aceton ist in allen Metriken in ppb angegeben.

In [ ]:
pipeline=Pipeline([('features',AutomaticSegments()),('scale',StandardScaler()),
                   ('select',FractionSelector()),('regression',PLSRegression(scale=False))])
searches={}; predictions={}; rows=[]
for method in ['pearson','rfe']:
    grid={'select__method':[method],'select__fraction':[.25,.5,1.], 'regression__n_components':[1,2]}
    search=GridSearchCV(pipeline,grid,cv=GroupKFold(3),scoring='neg_root_mean_squared_error',n_jobs=1,error_score='raise')
    search.fit(X,y,groups=groups); searches[method]=search
    predictions[method]=search.predict(splits['val']['X']).ravel()
    rows.append({'Methode':method,'CV_RMSE_ppb':-search.best_score_,
                 'Segmente':search.best_estimator_.named_steps['features'].n_segments_,
                 **regression_metrics(splits['val']['targets'][GAS],predictions[method])})
    print(method,search.best_params_)
show_results(rows)
fold_rows=[]
for fold,(train_idx,_) in enumerate(GroupKFold(3).split(X,y,groups),1):
    learned=AutomaticSegments().fit(X[train_idx])
    fold_rows.append({'Fold':fold,'Train_UGMs':len(np.unique(groups[train_idx])),'Segmente':learned.n_segments_})
show_results(fold_rows)
plot_comparison(splits['val']['targets'][GAS],predictions,'Aceton [ppb]: separate Validierung')

In [ ]:
winner=min(rows,key=lambda row:row['RMSE_ppb'])['Methode']; fitted=searches[winner].best_estimator_
fe=fitted.named_steps['features']; selected=fitted.named_steps['select'].indices_
plt.figure(figsize=(14,4)); plt.plot(time,X[0,0])
for segment in np.unique(selected//2):
    a,b=fe.boundaries_[segment:segment+2]; plt.axvspan(a/10,b/10,color='tab:orange',alpha=.3)
plt.title('Aceton: automatisch gelernte und ausgewaehlte Signalbereiche'); plt.xlabel('Zeit [s]'); plt.ylabel('Signal')
plt.tight_layout(); plt.show()
print('Gewaehlte Merkmale:',fe.get_feature_names_out()[selected].tolist())
final=[]
for name in ['test','test_extra']:
    truth=splits[name]['targets'][GAS]; pred=fitted.predict(splits[name]['X']).ravel()
    plot_comparison(truth,{winner:pred,'Trainingsmittelwert':np.full(len(truth),y.mean())},'Aceton: '+name)
    final.append({'Split':name,**regression_metrics(truth,pred)})
show_results(final)